# Full CTRNN Training Pipeline
Train an ensemble of CTRNNs on the **Perceptual** and **Context** Mante-style tasks. For each trained network we save its learning curve, scalar metrics, best-checkpoint weights, test-set hidden activations, and the signed stimulus coherences.

PID computation and analysis are handled in a separate notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import copy
import os
import json

from src.tasks.data_loader import load_mante_data
from src.models.ctrnn import CTRNN
from src.training.loss import compute_loss

In [ ]:
# ---------------------------------------------------------------------------
# Top-of-notebook parameters
# ---------------------------------------------------------------------------
TASKS = ['perceptual', 'context']  # options: ['perceptual'], ['context'], or ['perceptual', 'context']
N_p = 10          # number of CTRNNs to train on the perceptual task
N_c = 10          # number of CTRNNs to train on the context task
LOG_INTERVAL = 1  # log train loss/accuracy every this many gradient steps

In [ ]:
# Configuration (paths are relative to the notebooks/ directory, i.e. repo root via "..")
BASE_PATH = "../data/mante_style"
BATCH_SIZE = 1024 * 2
SUBSAMPLE_STEP = 10   # Subsample timesteps (simulate dt=10ms)
TEST_SPLIT = "test_uniform"  # held-out test set used for activation extraction

# Output directories (created per-task inside the training loop)
FIG_LC_DIR = "../figures/learning_curves"
RES_METRICS_DIR = "../results/accuracies_n_losses"
RES_WEIGHTS_DIR = "../results/model_weights"
RES_ACTS_DIR = "../results/model_activations"
RES_COHS_DIR = "../results/stimulus_coherences"

### 1. The Architecture
We wrap the existing `CTRNN` so that the forward pass returns `(outputs, hidden_states)`, with shapes `(Batch, Time, Classes)` and `(Batch, Time, Hidden)` respectively. The CTRNN re-initialises its hidden state to zero at the start of every `forward` call.

In [ ]:
class WrappedCTRNN(nn.Module):
    def __init__(self, input_dim, hidden_size, output_size=3):
        super().__init__()
        # Use the existing CTRNN
        self.model = CTRNN(input_size=input_dim, hidden_size=hidden_size, output_size=output_size)
        
    def forward(self, x):
        # CTRNN expects (Batch, Seq, Dim). 
        # We grab outputs and hidden_states, ignoring predictions.
        outputs, _, hidden_states = self.model(x, return_dynamics=True)
        return outputs, hidden_states

### 2. Loss & Accuracy
The loss is cross-entropy on the **final-timestep** output, provided by `compute_loss(..., condition="vanilla")` (i.e. `F.cross_entropy(outputs[:, -1, :], targets)`). The target for each trial is the label at the final timestep. Accuracy is the proportion of trials whose predicted class at the final timestep matches the ground-truth label.

In [ ]:
def final_step_accuracy(outputs, targets):
    """Proportion of trials where argmax(output[:, -1, :]) == target (final-timestep label)."""
    preds = outputs[:, -1, :].argmax(dim=1)
    return (preds == targets).float().mean().item()

### 3. Training, Testing & Plotting helpers
`train_ctrnn` trains a single network: it logs train loss/accuracy every `LOG_INTERVAL` gradient steps, computes validation loss/accuracy at the end of each epoch, and tracks the best checkpoint (lowest validation loss).

`test_ctrnn` runs the best checkpoint over the test set **one trial at a time**, zeroing the hidden state at the start of each trial, and collects the full hidden-state sequence and the signed (cued) coherence per trial.

`plot_learning_curve` draws the two-panel learning-curve figure.

In [ ]:
def train_ctrnn(model, train_loader, val_loader, device,
                num_epochs=50, lr=1e-3, patience=10, log_interval=LOG_INTERVAL):
    """
    Trains a single CTRNN.

    - Loss: cross-entropy on the final-timestep output (compute_loss, condition="vanilla").
    - Every `log_interval` gradient steps: record train loss and train accuracy.
    - End of each epoch: record validation loss and validation accuracy.
    - Best checkpoint = lowest validation loss (returned as `best_state`).
    Returns (history, best_state).
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    patience_counter = 0

    history = {
        'train_steps': [], 'train_loss': [], 'train_acc': [],
        'val_steps': [], 'val_loss': [], 'val_acc': [],
    }

    global_step = 0

    for epoch in range(num_epochs):
        # --- TRAINING PHASE ---
        model.train()
        for obs, labels, periods, cohs, ctxs in train_loader:
            obs = obs.to(device)
            targets = labels[:, -1].to(device)  # final-timestep label

            optimizer.zero_grad()
            outputs, _ = model(obs)
            loss = compute_loss(outputs, targets, condition="vanilla")
            loss.backward()
            optimizer.step()
            global_step += 1

            if global_step % log_interval == 0:
                history['train_steps'].append(global_step)
                history['train_loss'].append(loss.item())
                history['train_acc'].append(final_step_accuracy(outputs, targets))

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss_sum, val_acc_sum, n_trials = 0.0, 0.0, 0
        with torch.no_grad():
            for obs, labels, periods, cohs, ctxs in val_loader:
                obs = obs.to(device)
                targets = labels[:, -1].to(device)

                outputs, _ = model(obs)
                bs = obs.size(0)
                val_loss_sum += compute_loss(outputs, targets, condition="vanilla").item() * bs
                val_acc_sum += final_step_accuracy(outputs, targets) * bs
                n_trials += bs

        val_loss = val_loss_sum / n_trials
        val_acc = val_acc_sum / n_trials
        history['val_steps'].append(global_step)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # --- BEST-CHECKPOINT TRACKING (lowest val loss) ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1:03d}/{num_epochs}] | "
                  f"Train Loss: {history['train_loss'][-1]:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}. Best Val Loss: {best_val_loss:.4f}")
            break

    return history, best_state


def test_ctrnn(model, test_loader, device):
    """
    Runs the model over the test set one trial at a time (hidden state zeroed at the
    start of every trial). Returns:
      - hidden_activations: float32 array, shape [n_test_trials, T, n_hidden]
      - coherences:         float array,  shape [n_test_trials] (signed, cued stimulus)
      - test_loss, test_acc
    """
    model.eval()
    hidden_list = []
    coh_list = []
    loss_sum, correct, n_trials = 0.0, 0, 0

    with torch.no_grad():
        for obs, labels, periods, cohs, ctxs in test_loader:
            obs = obs.to(device)
            targets = labels[:, -1].to(device)

            for i in range(obs.size(0)):
                # A fresh forward pass zeros the CTRNN hidden state (trial-by-trial reset).
                xi = obs[i:i+1]
                ti = targets[i:i+1]
                outputs, hidden = model(xi)

                loss_sum += compute_loss(outputs, ti, condition="vanilla").item()
                correct += int((outputs[:, -1, :].argmax(dim=1) == ti).item())
                n_trials += 1

                hidden_list.append(hidden[0].cpu().numpy())   # (T, n_hidden)
                coh_list.append(float(cohs[i]))               # signed coherence of cued stimulus

    hidden_activations = np.stack(hidden_list).astype(np.float32)  # [n_test_trials, T, n_hidden]
    coherences = np.array(coh_list, dtype=float)                   # [n_test_trials]
    test_loss = loss_sum / n_trials
    test_acc = correct / n_trials
    return hidden_activations, coherences, test_loss, test_acc


def plot_learning_curve(history, save_path):
    """Two stacked subplots: (1) train/val loss, (2) train/val accuracy, vs gradient step."""
    fig, (ax_loss, ax_acc) = plt.subplots(2, 1, figsize=(8, 8))

    ax_loss.plot(history['train_steps'], history['train_loss'], label='Train loss')
    ax_loss.plot(history['val_steps'], history['val_loss'], label='Val loss')
    ax_loss.set_xlabel('Gradient step', fontsize=16)
    ax_loss.set_ylabel('Loss', fontsize=16)

    ax_acc.plot(history['train_steps'], history['train_acc'], label='Train acc')
    ax_acc.plot(history['val_steps'], history['val_acc'], label='Val acc')
    ax_acc.set_xlabel('Gradient step', fontsize=16)
    ax_acc.set_ylabel('Accuracy', fontsize=16)

    for ax in (ax_loss, ax_acc):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(labelsize=12)
        ax.legend(fontsize=14)

    fig.tight_layout()
    fig.savefig(save_path, dpi=300)
    plt.show()
    plt.close(fig)

In [ ]:
# 7 input channels, 100 hidden units, 3 output choices (Fixate, Choice 1, Choice 2)
input_size = 7 
hidden_size = 100
output_size = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device.type.upper()}")
if device.type == "cuda":
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
N_MODELS = {'perceptual': N_p, 'context': N_c}

for task in TASKS:
    n_models = N_MODELS[task]
    print(f"\n{'='*70}\nTASK: {task.upper()}  ({n_models} CTRNNs)\n{'='*70}")

    # Create all output directories for this task
    for base in (FIG_LC_DIR, RES_METRICS_DIR, RES_WEIGHTS_DIR, RES_ACTS_DIR, RES_COHS_DIR):
        os.makedirs(os.path.join(base, task), exist_ok=True)

    # Data loaders for this task
    train_loader = load_mante_data(f'{BASE_PATH}/{task}/train.npz', batch_size=BATCH_SIZE, shuffle=True,  subsample_step=SUBSAMPLE_STEP)
    val_loader   = load_mante_data(f'{BASE_PATH}/{task}/val.npz',   batch_size=BATCH_SIZE, shuffle=False, subsample_step=SUBSAMPLE_STEP)
    test_loader  = load_mante_data(f'{BASE_PATH}/{task}/{TEST_SPLIT}.npz', batch_size=BATCH_SIZE, shuffle=False, subsample_step=SUBSAMPLE_STEP)

    for idx in range(n_models):
        name = f"CTRNN_{idx:02d}"
        print(f"\n--- {task} | {name} ---")

        # Train
        model = WrappedCTRNN(input_size, hidden_size, output_size).to(device)
        history, best_state = train_ctrnn(model, train_loader, val_loader, device)

        # Test using the best checkpoint (lowest val loss)
        model.load_state_dict(best_state)
        hidden_activations, coherences, test_loss, test_acc = test_ctrnn(model, test_loader, device)

        # Scalar metrics: train/val from the last epoch, test from the test run
        metrics = {
            'train_loss': history['train_loss'][-1],
            'val_loss':   history['val_loss'][-1],
            'test_loss':  test_loss,
            'train_acc':  history['train_acc'][-1],
            'val_acc':    history['val_acc'][-1],
            'test_acc':   test_acc,
        }

        # 1. Learning curve figure (saved + displayed inline)
        plot_learning_curve(history, os.path.join(FIG_LC_DIR, task, f"{name}.png"))

        # 2. Scalar metrics JSON
        with open(os.path.join(RES_METRICS_DIR, task, f"{name}.json"), 'w') as f:
            json.dump(metrics, f, indent=2)

        # 3. Best-checkpoint weights
        torch.save(best_state, os.path.join(RES_WEIGHTS_DIR, task, f"{name}.pt"))

        # 4. Hidden activations [n_test_trials, T, n_hidden]
        np.save(os.path.join(RES_ACTS_DIR, task, f"{name}.npy"), hidden_activations)

        # 5. Signed stimulus coherences [n_test_trials]
        np.save(os.path.join(RES_COHS_DIR, task, f"{name}.npy"), coherences)

        print(f"  train_loss={metrics['train_loss']:.4f} | val_loss={metrics['val_loss']:.4f} | test_loss={metrics['test_loss']:.4f}")
        print(f"  train_acc ={metrics['train_acc']:.4f} | val_acc ={metrics['val_acc']:.4f} | test_acc ={metrics['test_acc']:.4f}")

print("\nPipeline complete.")